# Árboles de Decisión y Bosques Aleatorios en Python

### Actualización: mayo 2026

Este notebook proporciona una introducción práctica a los **árboles de decisión** y los **bosques aleatorios** (*Random Forests*), utilizando conjuntos de datos disponibles en fuentes públicas.

---

**Contenido:**

1. [Árboles de decisión: Teoría](#1-arboles-de-decision)
2. [Práctica: Árboles de decisión con el dataset Titanic](#2-practica-arboles-de-decision)
3. [Bosques aleatorios: Teoría](#3-bosques-aleatorios)
4. [Práctica: Random Forest con el dataset de Diabetes](#4-practica-random-forest)
5. [Comparación de modelos y conclusiones](#5-comparacion-y-conclusiones)
6. [Ejercicios propuestos](#6-ejercicios-propuestos)


<a id="1-arboles-de-decision"></a>
## 1. Árboles de Decisión

### 1.1 ¿Qué es un árbol de decisión?

Un **árbol de decisión** es un modelo de **aprendizaje supervisado** que se utiliza tanto para **clasificación** como para **regresión**.

La idea central es crear un modelo que prediga el valor de una variable objetivo aprendiendo **reglas de decisión simples** inferidas a partir de las características de los datos.

### 1.2 Componentes de un árbol de decisión

Los árboles de decisión generan reglas de tipo *"if-else"* organizadas en una estructura jerárquica:

- **Nodo raíz**: El nodo superior del árbol, donde comienza la primera división.
- **Nodos internos**: Cada uno representa una prueba sobre un atributo.
- **Ramas**: Representan el resultado de cada prueba.
- **Nodos hoja (terminales)**: Contienen la predicción final (etiqueta de clase).

El algoritmo comienza en el nodo raíz y desciende por las ramas según los resultados de las pruebas hasta llegar a un nodo hoja.


### 1.3 Métricas de pureza

El paso clave en la construcción de un árbol de decisión es seleccionar el atributo que mejor divide los datos. La "mejor división" se mide mediante métricas de **pureza**:

#### Entropía de la información (usada por el criterio *entropy*)

$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

Donde $p_i$ es la proporción de elementos de la clase $i$ en el conjunto $S$. Un valor de entropía de **0** indica un nodo puro (todos los elementos pertenecen a la misma clase); un valor alto indica mayor desorden.

#### Índice de Gini (usada por el criterio *gini*)

$$Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$$

El índice de Gini mide la probabilidad de clasificar incorrectamente un elemento aleatorio. Un valor de **0** indica pureza perfecta.

#### ¿Cuándo usar cada uno?

| Criterio | Características |
|----------|----------------|
| **Gini** | Más rápido de calcular; tiende a aislar la clase más frecuente en su propia rama. Es el valor por defecto en scikit-learn. |
| **Entropy** | Produce árboles más balanceados; penaliza más las impurezas. Puede ser ligeramente más lento. |

En la práctica, ambos criterios suelen producir resultados muy similares.


### 1.4 Proceso de construcción

1. **Selección de características**: Se evalúa cada atributo y se selecciona el que produce la mejor división según la métrica elegida.
2. **Generación del árbol**: Se generan subnodos recursivamente de arriba hacia abajo.
3. **Poda (Pruning)**: Se simplifica el árbol para evitar el **sobreajuste** (*overfitting*).

#### ⚠️ El problema del sobreajuste

Un árbol de decisión sin restricciones puede crecer hasta memorizar los datos de entrenamiento, lo que produce:
- **Alta precisión en entrenamiento** (train accuracy)
- **Baja precisión en datos nuevos** (test accuracy)

Para combatirlo, se utilizan técnicas como:
- **Pre-poda**: Limitar la profundidad máxima (`max_depth`), el mínimo de muestras por hoja (`min_samples_leaf`), etc.
- **Post-poda**: Construir el árbol completo y luego eliminar las ramas que no mejoran el rendimiento en un conjunto de validación.


### 1.5 Aplicaciones de los árboles de decisión

- **Diagnóstico médico**: Diagnosticar condiciones basadas en síntomas.
- **Crédito bancario**: Evaluar si un solicitante es apto para un crédito.
- **Gestión de riesgos**: Identificar factores de riesgo en proyectos.
- **Marketing dirigido**: Segmentar clientes para campañas personalizadas.
- **Detección de fraudes**: Identificar transacciones sospechosas.

> **Los árboles de decisión son también la base para modelos más poderosos como los Random Forests**, que combinan múltiples árboles para mejorar la precisión y la estabilidad del modelo.


<a id="2-practica-arboles-de-decision"></a>
## 2. Práctica: Árboles de Decisión con el Dataset Titanic

Utilizaremos **scikit-learn**, la biblioteca de aprendizaje automático más popular en Python.

### 2.1 Instalación de paquetes

Ejecuta la siguiente celda si necesitas instalar los paquetes:


In [ ]:
# Descomenta la siguiente línea si necesitas instalar los paquetes
# !pip install numpy pandas scikit-learn matplotlib seaborn imblearn

### 2.2 Importación de librerías

In [ ]:
#
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)
from imblearn.under_sampling import RandomUnderSampler
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid', context='notebook')
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente ✓")

### 2.3 Carga y exploración de datos

Utilizaremos el conjunto de datos del **Titanic**, con información de los pasajeros del barco.

| Variable | Descripción |
|----------|-------------|
| **Survived** | Sobrevivió al hundimiento (0 = No, 1 = Sí) |
| **Pclass** | Clase del pasajero (1, 2, 3) |
| **Name** | Nombre completo |
| **Sex** | Género (male/female) |
| **Age** | Edad en años |
| **Siblings/Spouses Aboard** | Hermanos/cónyuges a bordo |
| **Parents/Children Aboard** | Padres/hijos a bordo |
| **Fare** | Tarifa pagada |


In [ ]:
# Cargamos el dataset
titanic = pd.read_csv('https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv', sep=',')
print(f"Dimensiones del dataset: {titanic.shape}")
titanic.head()

In [ ]:
# Información general del dataset
titanic.info()

### 2.4 Limpieza y preparación de datos

In [ ]:
# Eliminamos las columnas que no usaremos para el modelo
titanic.drop(['Name', 'Fare'], axis=1, inplace=True)

# Renombramos columnas para simplicidad
titanic.columns = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'ParCh']

# Convertimos la variable categórica 'Sex' a numérica (1 = male, 0 = female)
titanic = pd.get_dummies(titanic, columns=['Sex'], drop_first=True)
titanic.rename(columns={'Sex_male': 'Sex'}, inplace=True)

# Reordenamos las columnas
titanic = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'ParCh']]

print("Dataset limpio:")
titanic.head()

### 2.5 Exploración de la variable objetivo

In [ ]:
# Distribución de la variable objetivo
print("Distribución de supervivencia:")
print(titanic.Survived.value_counts())
print()
print("Proporciones:")
print(titanic.Survived.value_counts(normalize=True).round(3))

# Visualización
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
titanic.Survived.value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_xticklabels(['No Sobrevivió (0)', 'Sobrevivió (1)'], rotation=0)
ax.set_ylabel('Cantidad')
ax.set_title('Distribución de la variable Survived')
plt.tight_layout()
plt.show()

### 2.6 Balanceo de datos

Observamos que las clases están **desbalanceadas**. Usaremos *undersampling* para igualar las proporciones antes de entrenar.


In [ ]:
# Separamos features (X) y variable objetivo (y)
X_titanic = titanic.drop('Survived', axis=1)
y_titanic = titanic.Survived

# Balanceamos los datos con undersampling
undersample = RandomUnderSampler(random_state=42)
X_bal, y_bal = undersample.fit_resample(X_titanic, y_titanic)

print("Proporciones después del balanceo:")
print(y_bal.value_counts(normalize=True).round(3))

### 2.7 División en conjuntos de entrenamiento y prueba

In [ ]:
# 70% entrenamiento, 30% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.30, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")

### 2.8 Entrenamiento con búsqueda de hiperparámetros

Usaremos `GridSearchCV` para encontrar la mejor combinación de hiperparámetros.


In [ ]:
# Definir el clasificador y los hiperparámetros a probar
clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Búsqueda exhaustiva con validación cruzada
grid_search_dt = GridSearchCV(clf, param_grid=param_grid, cv=10,
                               return_train_score=True, scoring='accuracy')
grid_search_dt.fit(X_train, y_train)

print("Mejores hiperparámetros encontrados:")
print(grid_search_dt.best_params_)
print(f"\nMejor accuracy en validación cruzada: {grid_search_dt.best_score_:.4f}")

In [ ]:
# Modelo con parámetros optimizados
best_dt = grid_search_dt.best_estimator_

### 2.9 Evaluación del modelo

In [ ]:
# Predicciones
y_train_pred = best_dt.predict(X_train)
y_test_pred = best_dt.predict(X_test)

# Accuracy
print("=" * 50)
print("EVALUACIÓN DEL ÁRBOL DE DECISIÓN - TITANIC")
print("=" * 50)
print(f"Accuracy en TRAIN: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Accuracy en TEST:  {accuracy_score(y_test, y_test_pred):.4f}")
print()

# Verificar overfitting
diff = accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_test_pred)
if diff > 0.05:
    print(f"⚠️  Diferencia train-test: {diff:.4f} → Posible sobreajuste")
else:
    print(f"✓  Diferencia train-test: {diff:.4f} → No hay sobreajuste evidente")

print()
print("Reporte de clasificación (TEST):")
print(classification_report(y_test, y_test_pred, target_names=['No Sobrevivió', 'Sobrevivió']))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
cm = confusion_matrix(y_test, y_test_pred, labels=best_dt.classes_)
ConfusionMatrixDisplay(cm, display_labels=['No Sobrevivió', 'Sobrevivió']).plot(ax=ax, cmap='Blues')
ax.set_title('Matriz de Confusión - Árbol de Decisión (Titanic)')
plt.tight_layout()
plt.show()

In [ ]:
# AUC-ROC
y_test_proba = best_dt.predict_proba(X_test)[:, 1]
auc_dt = roc_auc_score(y_test, y_test_proba)

fpr, tpr, _ = roc_curve(y_test, y_test_proba)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.plot(fpr, tpr, color='#3498db', lw=2, label=f'Árbol de Decisión (AUC = {auc_dt:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curva ROC - Árbol de Decisión (Titanic)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.10 Visualización del árbol

In [ ]:
# Visualización del árbol de decisión
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
plot_tree(best_dt,
          feature_names=X_train.columns,
          class_names=['No Sobrevivió', 'Sobrevivió'],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title('Árbol de Decisión Optimizado - Titanic', fontsize=14)
plt.tight_layout()
plt.show()

### 2.11 Importancia de las características

In [ ]:
# Importancia de features
feature_imp_dt = pd.Series(best_dt.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
sns.barplot(x=feature_imp_dt.values, y=feature_imp_dt.index, ax=ax, palette='viridis')
for i, v in enumerate(feature_imp_dt.values):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
ax.set_title('Importancia de características - Árbol de Decisión (Titanic)')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print("Interpretación:")
print(f"  → {feature_imp_dt.index[0]} es el factor más importante para predecir la supervivencia.")
print(f"  → {feature_imp_dt.index[-1]} tiene menor impacto y podría eliminarse sin afectar mucho al modelo.")

<a id="3-bosques-aleatorios"></a>
## 3. Introducción a Random Forest

### 3.1 ¿Qué es un Random Forest?

Random Forest es un **algoritmo de ensamble** que combina múltiples árboles de decisión para producir una predicción más precisa y robusta.

> **Ensamble**: Técnica que combina múltiples modelos para mejorar la precisión y la estabilidad. En Random Forest, cada modelo del ensamble es un árbol de decisión.

### 3.2 ¿Cómo funciona?

1. **Bagging (Bootstrap Aggregating)**: Se toman múltiples muestras aleatorias **con reemplazo** de los datos de entrenamiento. Cada muestra se usa para entrenar un árbol diferente.
2. **Selección aleatoria de features**: En cada nodo de cada árbol, solo se considera un subconjunto aleatorio de las características disponibles para la división. Esto reduce la correlación entre árboles.
3. **Agregación de predicciones**:
   - **Clasificación**: La predicción final es la clase con la **mayoría de votos** entre todos los árboles.
   - **Regresión**: La predicción final es el **promedio** de las predicciones de todos los árboles.

### 3.3 Ventajas sobre un solo árbol de decisión

| Aspecto | Árbol de Decisión | Random Forest |
|---------|-------------------|---------------|
| Sobreajuste | Alto riesgo | Mucho menor |
| Varianza | Alta | Baja (por promediado) |
| Interpretabilidad | Alta | Media |
| Precisión | Menor | Generalmente mayor |
| Tiempo de entrenamiento | Rápido | Más lento |


<a id="4-practica-random-forest"></a>
## 4. Práctica: Random Forest con el Dataset de Diabetes

Utilizaremos el dataset **Pima Indians Diabetes** de [Kaggle](https://www.kaggle.com/datasets/kumargh/pimaindiansdiabetescsv).

### 4.1 Descripción del dataset

El conjunto de datos contiene información médica de mujeres Pima Indian de Arizona que participaron en un estudio sobre diabetes.

| Variable | Descripción |
|----------|-------------|
| **Pregnancies** | Número de embarazos |
| **Glucose** | Concentración de glucosa en plasma (2h en prueba de tolerancia) |
| **BloodPressure** | Presión arterial diastólica (mm Hg) |
| **SkinThickness** | Grosor del pliegue cutáneo tricipital (mm) |
| **Insulin** | Concentración de insulina en suero (2h, mu U/ml) |
| **BMI** | Índice de masa corporal |
| **DiabetesPedigreeFunction** | Función de diabetes basada en antecedentes familiares |
| **Age** | Edad (años) |
| **Outcome** | Variable objetivo (1 = diabetes positivo, 0 = negativo) |


### 4.2 Carga y exploración de datos

In [ ]:
# Carga correcta del dataset (el CSV tiene header)
df_diabetes = pd.read_csv('diabetes.csv')

print(f"Dimensiones del dataset: {df_diabetes.shape}")
print(f"\nTipos de datos:")
print(df_diabetes.dtypes)
print(f"\nPrimeras filas:")
df_diabetes.head()

In [ ]:
# Verificar datos duplicados
duplicados = df_diabetes.duplicated().sum()
print(f"Registros duplicados: {duplicados}")
df_diabetes.drop_duplicates(inplace=True)
print(f"Dimensiones tras eliminar duplicados: {df_diabetes.shape}")

In [ ]:
# Variables con posibles datos perdidos (valores 0 donde no deberían ser 0)
cols_con_ceros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
missing_info = (df_diabetes[cols_con_ceros] == 0).sum().reset_index()
missing_info.columns = ['Variable', 'Ceros_sospechosos']
missing_info['Porcentaje'] = (missing_info['Ceros_sospechosos'] / len(df_diabetes) * 100).round(1)
print("Variables con valores 0 sospechosos (posibles datos faltantes):")
missing_info

In [ ]:
# Distribución de la variable objetivo
print("Distribución de la variable Outcome:")
print(df_diabetes.Outcome.value_counts())
print()
print("Proporciones:")
print(df_diabetes.Outcome.value_counts(normalize=True).round(3))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
labels = ['Negativo (0)', 'Positivo (1)']
colors = ['#2ecc71', '#e74c3c']
df_diabetes.Outcome.value_counts().plot(kind='bar', ax=ax, color=colors)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel('Cantidad')
ax.set_title('Distribución de diabetes en el dataset')
plt.tight_layout()
plt.show()

### 4.3 Preparación de datos y entrenamiento

In [ ]:
# Separar features (X) y variable objetivo (y)
X_diabetes = df_diabetes.drop('Outcome', axis=1)
y_diabetes = df_diabetes['Outcome']

# División: 80% entrenamiento, 20% prueba
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes, y_diabetes, test_size=0.20, random_state=42
)

print(f"Datos de entrenamiento: {X_train_d.shape[0]} muestras")
print(f"Datos de prueba: {X_test_d.shape[0]} muestras")

In [ ]:
# Definir el modelo y los hiperparámetros a explorar
rfc = RandomForestClassifier(random_state=42)

# NOTA: Solo usamos criterios válidos para CLASIFICACIÓN (gini, entropy, log_loss)
param_grid_rf = {
    'n_estimators': [10, 25, 50, 100],
    'max_depth': [5, 10, 15],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 4],
}

print(f"Total de combinaciones a probar: {np.prod([len(v) for v in param_grid_rf.values()])}")

In [ ]:
# Búsqueda de hiperparámetros con validación cruzada
grid_search_rf = GridSearchCV(
    estimator=rfc, param_grid=param_grid_rf,
    cv=5, scoring='accuracy', n_jobs=-1
)

grid_search_rf.fit(X_train_d, y_train_d)

print("Mejores hiperparámetros encontrados:")
for k, v in grid_search_rf.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nMejor accuracy en validación cruzada: {grid_search_rf.best_score_:.4f}")

In [ ]:
# Mejor modelo
best_rf = grid_search_rf.best_estimator_

### 4.4 Evaluación del modelo

In [ ]:
# Predicciones
y_train_pred_d = best_rf.predict(X_train_d)
y_test_pred_d = best_rf.predict(X_test_d)

# Métricas
print("=" * 50)
print("EVALUACIÓN DEL RANDOM FOREST - DIABETES")
print("=" * 50)
acc_train = accuracy_score(y_train_d, y_train_pred_d)
acc_test = accuracy_score(y_test_d, y_test_pred_d)
print(f"Accuracy en TRAIN: {acc_train:.4f}")
print(f"Accuracy en TEST:  {acc_test:.4f}")
print()

# Verificar overfitting
diff = acc_train - acc_test
if diff > 0.05:
    print(f"⚠️  Diferencia train-test: {diff:.4f} → Posible sobreajuste")
else:
    print(f"✓  Diferencia train-test: {diff:.4f} → No hay sobreajuste evidente")

print()
print("Reporte de clasificación (TEST):")
print(classification_report(y_test_d, y_test_pred_d, target_names=['No Diabetes', 'Diabetes']))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
cm = confusion_matrix(y_test_d, y_test_pred_d)
ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes']).plot(ax=ax, cmap='Greens')
ax.set_title('Matriz de Confusión - Random Forest (Diabetes)')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
y_test_proba_d = best_rf.predict_proba(X_test_d)[:, 1]
auc_rf = roc_auc_score(y_test_d, y_test_proba_d)

fpr_rf, tpr_rf, _ = roc_curve(y_test_d, y_test_proba_d)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.plot(fpr_rf, tpr_rf, color='#27ae60', lw=2, label=f'Random Forest (AUC = {auc_rf:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curva ROC - Random Forest (Diabetes)')
ax.legend()
plt.tight_layout()
plt.show()

### 4.5 Validación cruzada

In [ ]:
# Validación cruzada K-Fold
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=42)
scores = cross_val_score(best_rf, X_train_d, y_train_d, cv=cv, scoring='accuracy')

print(f"Validación cruzada con K={k}:")
print(f"  Accuracy por fold: {[f'{s:.4f}' for s in scores]}")
print(f"  Accuracy promedio: {scores.mean():.4f} ± {scores.std():.4f}")

### 4.6 Importancia de las características

In [ ]:
# Importancia de features
feature_imp_rf = pd.Series(best_rf.feature_importances_, index=X_train_d.columns).sort_values(ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.barplot(x=feature_imp_rf.values, y=feature_imp_rf.index, ax=ax, palette='YlGn_r')
for i, v in enumerate(feature_imp_rf.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
ax.set_title('Importancia de características - Random Forest (Diabetes)')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print("Interpretación:")
print(f"  → {feature_imp_rf.index[0]} es el factor más importante para predecir diabetes.")
print(f"  → {feature_imp_rf.index[-1]} tiene el menor impacto en la predicción.")

### 4.7 Visualización de un árbol del bosque

In [ ]:
# Visualizamos uno de los árboles del Random Forest
fig, ax = plt.subplots(1, 1, figsize=(25, 12))
plot_tree(best_rf.estimators_[0],
          feature_names=X_train_d.columns,
          class_names=['No Diabetes', 'Diabetes'],
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Uno de los árboles del Random Forest - Diabetes', fontsize=14)
plt.tight_layout()
plt.show()

<a id="5-comparacion-y-conclusiones"></a>
## 5. Comparación de Modelos y Conclusiones

### 5.1 Resumen de resultados


In [ ]:
# Tabla comparativa
print("=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)
print()
print(f"{'Métrica':<30} {'Decision Tree':<15} {'Random Forest':<15}")
print(f"{'Dataset':<30} {'Titanic':<15} {'Diabetes':<15}")
print("-" * 60)
print(f"{'Accuracy (Train)':<30} {accuracy_score(y_train, y_train_pred):<15.4f} {acc_train:<15.4f}")
print(f"{'Accuracy (Test)':<30} {accuracy_score(y_test, y_test_pred):<15.4f} {acc_test:<15.4f}")
print(f"{'AUC-ROC (Test)':<30} {auc_dt:<15.4f} {auc_rf:<15.4f}")
print(f"{'Diferencia Train-Test':<30} {accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_test_pred):<15.4f} {diff:<15.4f}")
print()
print("Nota: Los modelos se aplicaron a datasets diferentes,")
print("por lo que la comparación directa de accuracy no es apropiada.")
print("Lo importante es observar la diferencia train/test de cada modelo.")

### 5.2 Conclusiones

1. **Árboles de decisión** son modelos intuitivos y fáciles de interpretar, pero son propensos al **sobreajuste** si no se controlan sus hiperparámetros (profundidad, mínimo de muestras, etc.).

2. **Random Forest** reduce el riesgo de sobreajuste al combinar múltiples árboles con muestreo aleatorio (*bagging*) y selección aleatoria de features, a costa de perder algo de interpretabilidad.

3. La **búsqueda de hiperparámetros** (`GridSearchCV`) es una herramienta esencial para optimizar el rendimiento de ambos modelos.

4. Para evaluar un modelo de clasificación no basta con el **accuracy**: es importante analizar también el **precision**, **recall**, **F1-score** y **AUC-ROC**, especialmente cuando las clases están desbalanceadas.

5. La **importancia de features** nos ayuda a entender qué variables contribuyen más a las predicciones, lo cual es valioso para la interpretación del modelo.


<a id="6-ejercicios-propuestos"></a>
## 6. Ejercicios Propuestos

### Ejercicio 1: Experimentar con hiperparámetros
Modifica el `max_depth` del árbol de decisión a valores de 1 a 10 y grafica cómo cambia el accuracy en train y test. ¿A partir de qué profundidad comienza el sobreajuste?

### Ejercicio 2: Reemplazar valores faltantes
En el dataset de diabetes, los valores 0 en columnas como `Glucose`, `BloodPressure`, `BMI`, etc., son probablemente datos faltantes. Reemplázalos por la mediana de cada columna y vuelve a entrenar el Random Forest. ¿Mejora el rendimiento?

### Ejercicio 3: Comparar en el mismo dataset
Entrena tanto un árbol de decisión como un Random Forest en el dataset de diabetes (o el de Titanic) y compara sus métricas directamente. ¿Cuánto mejora el Random Forest sobre un solo árbol?

### Ejercicio 4: Aumentar el número de árboles
Varía `n_estimators` de 5 a 200 en el Random Forest y grafica cómo evoluciona el accuracy en test. ¿Hay un punto a partir del cual más árboles no mejoran el resultado?

### Ejercicio 5: Curva de aprendizaje
Investiga la función `learning_curve` de scikit-learn y úsala para visualizar cómo cambia el rendimiento del modelo a medida que se incrementan los datos de entrenamiento.
